In [1]:
!pip install selenium
!pip install webdriver-manager

In [2]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable

OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,215 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 https://ppa.launc

In [9]:
from getpass import getpass

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

PROFILE_URL = "https://terakoya.sejuku.net/register"
INTRO_TEXT = "プログラミング学習中です！今はスクレイピングに挑戦しています！"

# あなたが指定したCSSセレクタ
ICON_BTN = "#root > div.sc-kXHtdE.cPbGym > div > div > header > div.sc-bNShyZ.jLAYEi > div.sc-flkahu.iqyAOo > div"
EDIT_BTN = "#root > div.sc-eqgsLa.hPeIN > div > div > main > div > div.sc-bVsQZF.eQwtkq > button"
INTRO_TEXTAREA = "#root > div.sc-eqgsLa.hPeIN > div > div > main > div > div.sc-bVsQZF.eQwtkq > div:nth-child(10) > div.sc-jsbVYh.ljsGeL > textarea"
UPDATE_BTN = "#root > div.sc-eqgsLa.hPeIN > div > div > main > div > div.sc-bVsQZF.eQwtkq > button.sc-dTvVRJ.bHRgJQ.sc-lfzvHA.kmNVuB"


def main():
    email = input("SAMURAI TERAKOYAのログインメールアドレス: ").strip()
    password = getpass("SAMURAI TERAKOYAのパスワード（非表示）: ").strip()

    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    # 画面を見ながら確認したい場合は headless を付けない（下はOFFのまま）
    # options.add_argument("--headless=new")

    # ChromeDriverManagerを使用してChromeDriverを自動的に管理
    s = Service(ChromeDriverManager().install())

    # Google Chrome用のブラウザドライバーインスタンスを作成
    driver = webdriver.Chrome(service=s, options=options)
    wait = WebDriverWait(driver, 15)

    try:
        # 1) プロフィールページへ（未ログインならログイン画面が表示される想定）
        driver.get(PROFILE_URL)

        # ログインフォームの親要素が表示されるまで待機
        header_login_button  = wait.until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, '#root > header > div'))
        )

        header_login_button.click()


        # 2) ログイン（画面要素は一般的なCSSで待つ）
        driver.save_screenshot('screenshot.png')
        parent_element = driver.find_element(By.CSS_SELECTOR, '#root > div.sc-kNOymR.TvzZn > div.sc-lgpSej.gsAReM > div.sc-dntSTA.dQOsDP > div')
        email_input = parent_element.find_element(By.NAME, 'email')
        password_input = parent_element.find_element(By.NAME, 'password')
        driver.save_screenshot('screenshot2.png')

        email_input.clear()
        email_input.send_keys(email)
        password_input.clear()
        password_input.send_keys(password)
        driver.save_screenshot('screenshot3.png')
        login_btn = wait.until(
            EC.visibility_of_element_located(
                (By.CSS_SELECTOR,
                '#root > div.sc-kNOymR.TvzZn > div.sc-lgpSej.gsAReM > div.sc-dntSTA.dQOsDP > button')
            )
        )
        driver.save_screenshot('screenshot4.png')
        login_btn.click()


        # 3) プロフィール（アカウント設定）画面に来るまで待つ
        icon_btn = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "#root > div.sc-kXHtdE.cPbGym > div > div > header > div.sc-bNShyZ.jLAYEi > div.sc-flkahu.iqyAOo > div")))
        driver.save_screenshot("01_after_login.png")
        icon_btn.click()

        #アカウント設定をクリックする
        account_btn = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "#root > div.sc-kXHtdE.cPbGym > div > div > header > div.sc-hPSbXP.jFIBPk > ul > a:nth-child(1) > li")))
        driver.save_screenshot("01_after_login2.png")
        account_btn.click()

       #編集ボタンをクリックする
        setting_btn = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "#root > div.sc-eqgsLa.hPeIN > div > div > main > div > div.sc-bVsQZF.eQwtkq > button")))
        driver.save_screenshot("01_after_login3.png")
        setting_btn .click()

        # 4) 自己紹介欄を特定して更新
        prf_box = wait.until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, '#root > div.sc-eqgsLa.hPeIN > div > div > main > div > div.sc-bVsQZF.eQwtkq > div:nth-child(10) > div.sc-jsbVYh.ljsGeL > textarea'))
        )

        prf_box.send_keys("プログラミング学習中です！今はスクレイピングに挑戦しています！")
        driver.save_screenshot("02_filled_intro.png")

        # 5) 保存ボタンをクリック
        save_btn = wait.until(
            EC.visibility_of_element_located((
                By.CSS_SELECTOR,
                '#root > div.sc-eqgsLa.hPeIN > div > div > main > div > div.sc-bVsQZF.eQwtkq > button.sc-dTvVRJ.bHRgJQ.sc-lfzvHA.kmNVuB'
            ))
        )
        save_btn.click()

        # 6) 保存完了っぽい表示（トースト等）が出るまで待つ（無ければbody待ちでもOK）
        # 実際の文言に合わせて調整してください
        try:
            wait.until(
                EC.visibility_of_element_located((
                    By.CSS_SELECTOR,
                    '[class*="toast"], [role="alert"], .notification'
                ))
            )
        except Exception:
            # 完了表示が取れない場合でも、少し待ってスクショ
            wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "body")))

        driver.save_screenshot("03_saved.png")
        print("更新が完了しました。スクリーンショット: 01_after_login.png / 02_filled_intro.png / 03_saved.png")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

SAMURAI TERAKOYAのログインメールアドレス: yuchan.0w0.opp@gmail.com
SAMURAI TERAKOYAのパスワード（非表示）: ··········
更新が完了しました。スクリーンショット: 01_after_login.png / 02_filled_intro.png / 03_saved.png
